# W&B Sweep — Proximal Policy Optimization (PPO)
Búsqueda de hiperparámetros para el agente PPO en el ambiente Simple (CSTR).
- Método: Random Search
- Proyecto W&B: `Tesis_PPO_CTRL`
- Arquitectura: simple (solo CTRL)
- Acción: continua

## 1. Instalación e Imports

In [ ]:
import os
import random
import numpy as np
import torch
import wandb
import sys
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Clonar desde Github:
!git clone https://github.com/valeriaeskenazi/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning.git
PROJECT_PATH = '/content/Control-PID-Adaptativo-Inteligente-mediante-Reinforcement-Learning/Version_4'
sys.path.append(PROJECT_PATH)

In [ ]:
from Environment.Simulation_Env.Reactor_CSTR import CSTRSimulator
from Environment.PIDControlEnv_simple import PIDControlEnv_Simple
from Environment.PIDControlEnv_complex import PIDControlEnv_Complex
from Agente.PPO.train_PPO import PPOTrainer
from Agente.PPO.algorithm_PPO import PPOAgent
from Aux.Plots import SimplePlotter, print_summary

print('Imports completados')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {"CUDA" if torch.cuda.is_available() else "CPU"}')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

## 2. Login W&B

In [ ]:
!pip install wandb --quiet

In [ ]:
wandb.login()

## 3. Configuración del Sweep

In [ ]:
# ============ LISTAS DE OPCIONES PREDEFINIDAS ============

REWARD_WEIGHTS_OPTIONS = [
    {'error': 1.0, 'tiempo': 0.3,   'overshoot': 0.2,  'energy': 0.1},    # 0: balanceado
    {'error': 2.0, 'tiempo': 0.1,   'overshoot': 0.5,  'energy': 0.1},    # 1: foco en error y overshoot
    {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3,  'energy': 0.001},  # 2: config base
    {'error': 3.0, 'tiempo': 0.1,   'overshoot': 0.1,  'energy': 0.05},   # 3: solo error importa
]

HIDDEN_DIMS_OPTIONS = [
    (64, 32),
    (128, 64),
    (128, 128, 64),
    (256, 128, 64),
]

# ============ FIJOS PARA TODOS LOS RUNS ============
SEED                         = 42
N_EPISODES                   = 300
EVAL_FREQUENCY               = 50
EARLY_STOPPING_PATIENCE      = 10
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
N_MANIPULABLE_VARS           = 2
MANIPULABLE_RANGES           = [(300, 420), (99.5, 104)]
DT                           = 1.0
DEVICE                       = 'cuda' if torch.cuda.is_available() else 'cpu'

WANDB_TEAM    = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_PPO_CTRL'

sweep_config = {
    'name':   'ppo_cstr_random_search',
    'method': 'random',

    'metric': {
        'name': 'eval_reward',
        'goal': 'maximize'
    },

    'parameters': {

        # ============ AMBIENTE ============
        'max_time_detector':  {'values': [15, 30, 60]},
        'max_steps':          {'values': [20, 50, 100]},
        'reward_dead_band':   {'values': [0.01, 0.02, 0.05]},
        'delta_percent_ctrl': {'values': [0.05, 0.1, 0.2, 0.3]},
        'reward_weights_idx': {'values': [0, 1, 2, 3]},

        # ============ CRITERIOS DE ESTABILIDAD ============
        'error_increase_tolerance': {'values': [1.2, 1.5, 2.0]},
        'max_sign_changes_ratio':   {'values': [0.1, 0.2, 0.3]},
        'max_abrupt_change_ratio':  {'values': [0.03, 0.05, 0.1]},
        'abrupt_change_threshold':  {'values': [0.2, 0.3, 0.5]},

        # ============ AGENTE PPO ============
        # Arquitectura (igual que AC para comparación justa)
        'hidden_dims_idx': {'values': [0, 1, 2, 3]},
        'lr_actor':        {'values': [0.00001, 0.0001, 0.0003]},
        'lr_critic':       {'values': [0.0001,  0.001,  0.01]},
        'gamma':           {'values': [0.95, 0.99, 0.999]},

        # Hiperparámetros específicos de PPO
        'clip_epsilon':    {'values': [0.1, 0.2, 0.3]},       # rango típico PPO
        'ppo_epochs':        {'values': [4, 8, 16]},             # epochs por update
        'gae_lambda':      {'values': [0.9, 0.95, 0.99]},      # GAE lambda
        'entropy_coef':    {'values': [0.001, 0.01, 0.05]},
        'value_coef':      {'values': [0.5, 1.0]},             # peso del critic en la loss
        'mini_batch_size':      {'values': [32, 64, 128]},
        'rollout_steps':   {'values': [256, 512, 1024]},        # pasos antes de cada update
    }
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_TEAM)
print(f'Sweep creado: {sweep_id}')

## 4. Función del Sweep

In [ ]:
def sweep_run():
    # -------- Inicializar run --------
    wandb.init()
    cfg = wandb.config

    # -------- Reproducibilidad --------
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    wandb.config.update({'seed': SEED}, allow_val_change=True)

    # -------- Resolver índices --------
    reward_weights = REWARD_WEIGHTS_OPTIONS[cfg.reward_weights_idx]
    hidden_dims    = HIDDEN_DIMS_OPTIONS[cfg.hidden_dims_idx]

    wandb.config.update({
        'reward_weights': str(reward_weights),
        'hidden_dims':    str(hidden_dims),
    }, allow_val_change=True)

    # -------- Configurar CSTR --------
    cstr = CSTRSimulator(
        dt=DT,
        control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
    )

    # -------- Construir config del trainer --------
    trainer_config = {
        # === AMBIENTE ===
        'env_config': {
            'architecture':          'simple',
            'env_type':              'simulation',
            'action_type':           'continuous',
            'n_manipulable_vars':    N_MANIPULABLE_VARS,
            'manipulable_ranges':    MANIPULABLE_RANGES,
            'manipulable_setpoints': None,
            'dt_usuario':            DT,
            'max_steps':             cfg.max_steps,
            'max_time_detector':     cfg.max_time_detector,
            'reward_dead_band':      cfg.reward_dead_band,
            'delta_percent_ctrl':    cfg.delta_percent_ctrl,
            'reward_weights':        reward_weights,
            'pid_limits': [
                (0.01, 50.0),
                (0.001, 1.0),
                (0.0,   1.0)
            ],
            'agent_controller_config': {'agent_type': 'continuous'},
            'env_type_config': {
                'dt': DT,
                'control_limits': (MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1])
            },
            'stability_config': {
                'error_increase_tolerance': cfg.error_increase_tolerance,
                'max_sign_changes_ratio':   cfg.max_sign_changes_ratio,
                'max_abrupt_change_ratio':  cfg.max_abrupt_change_ratio,
                'abrupt_change_threshold':  cfg.abrupt_change_threshold,
            },
        },

        # === AGENTE PPO ===
        'agent_ctrl_config': {
            'algorithm':     'ppo',
            'state_dim':     N_MANIPULABLE_VARS * 5,
            'action_dim':    N_MANIPULABLE_VARS * 3,
            'n_vars':        N_MANIPULABLE_VARS,
            'action_type':   'continuous',
            'hidden_dims':   hidden_dims,
            'lr_actor':      cfg.lr_actor,
            'lr_critic':     cfg.lr_critic,
            'gamma':         cfg.gamma,
            'clip_epsilon':  cfg.clip_epsilon,
            'ppo_epochs':      cfg.n_epochs,
            'gae_lambda':    cfg.gae_lambda,
            'entropy_coef':  cfg.entropy_coef,
            'value_coef':    cfg.value_coef,
            'mini_batch_size':    cfg.batch_size,
            'rollout_steps': cfg.rollout_steps,
            'device':        DEVICE,
            'seed':          SEED,
        },

        # === ENTRENAMIENTO ===
        'n_episodes':                    N_EPISODES,
        'eval_frequency':                EVAL_FREQUENCY,
        'save_frequency':                9999,
        'log_frequency':                 50,
        'checkpoint_dir':                f'checkpoints/ppo_{wandb.run.name}',
        'early_stopping_patience':       EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta_pct':  EARLY_STOPPING_MIN_DELTA_PCT,
        'use_wandb': True,
    }

    # -------- Conectar CSTR al ambiente --------
    trainer = PPOTrainer(trainer_config)
    trainer.env.proceso.connect_external_process(cstr)

    # -------- Entrenar --------
    trainer.train()

    # -------- Métricas finales del run --------
    wandb.log({
        'final_eval_reward':        trainer.best_reward,
        'total_episodes':           len(trainer.episode_rewards),
        'final_reward_mean10':      np.mean(trainer.episode_rewards[-10:]),
        'final_energy_mean10':      np.mean(trainer.episode_energies[-10:]),
        'final_overshoot_mean10':   np.mean(trainer.episode_max_overshoots[-10:]),
        'final_actor_loss_mean10':  np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
        'final_critic_loss_mean10': np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
        'final_clip_frac_mean10':   np.mean(trainer.clip_fractions[-10:]) if hasattr(trainer, 'clip_fractions') and trainer.clip_fractions else 0,
    })

    print(f'Run completado: {wandb.run.name}')
    wandb.finish()

## 5. Lanzar Sweep

In [ ]:
wandb.agent(sweep_id, function=sweep_run, count=30)

## 6. Run de Producción (post-sweep)

Reemplazá los hiperparámetros con los del mejor run del sweep antes de correr esta celda.

In [ ]:
# ============ CONFIG RUN PRINCIPAL PPO 15K ============
# Actualizar con los mejores hiperparámetros del sweep

WANDB_ENTITY  = 've326684-universidad-ort-uruguay'
WANDB_PROJECT = 'Tesis_PPO_CTRL_PROD'
RUN_NAME      = 'ppo_ctrl_15k_prod'

N_EPISODES                   = 15000
EVAL_FREQUENCY               = 100
LOG_FREQUENCY                = 100
SAVE_FREQUENCY               = 2000
EARLY_STOPPING_PATIENCE      = 40
EARLY_STOPPING_MIN_DELTA_PCT = 0.01
SEED                         = 42

trainer_config = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : 2,
        'manipulable_ranges'     : [(300, 420), (99.5, 104)],
        'manipulable_setpoints'  : None,
        'dt_usuario'             : 1.0,
        'max_steps'              : 100,         # ← del sweep
        'max_time_detector'      : 60,          # ← del sweep
        'reward_dead_band'       : 0.02,        # ← del sweep
        'delta_percent_ctrl'     : 0.2,         # ← del sweep
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {'dt': 1.0, 'control_limits': ((300, 420), (99.5, 104))},
        'stability_config'       : {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
    },
    'agent_ctrl_config': {
        'algorithm'    : 'ppo',
        'state_dim'    : 10,
        'action_dim'   : 6,
        'n_vars'       : 2,
        'action_type'  : 'continuous',
        'hidden_dims'  : (64, 32),        # ← del sweep
        'lr_actor'     : 3e-04,           # ← del sweep
        'lr_critic'    : 1e-03,           # ← del sweep
        'gamma'        : 0.99,            # ← del sweep
        'clip_epsilon' : 0.2,             # ← del sweep (0.2 es el default clásico)
        'ppo_epochs'     : 8,               # ← del sweep
        'gae_lambda'   : 0.95,            # ← del sweep
        'entropy_coef' : 0.01,            # ← del sweep
        'value_coef'   : 0.5,             # ← del sweep
        'mini_batch_size'   : 64,              # ← del sweep
        'rollout_steps': 512,             # ← del sweep
        'device'       : 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed'         : SEED,
    },
    'n_episodes'                  : N_EPISODES,
    'eval_frequency'              : EVAL_FREQUENCY,
    'log_frequency'               : LOG_FREQUENCY,
    'save_frequency'              : SAVE_FREQUENCY,
    'checkpoint_dir'              : f'checkpoints/{RUN_NAME}',
    'early_stopping_patience'     : EARLY_STOPPING_PATIENCE,
    'early_stopping_min_delta_pct': EARLY_STOPPING_MIN_DELTA_PCT,
    'use_wandb': True,
}

# ============ REPRODUCIBILIDAD ============
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# ============ INIT WANDB ============
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = RUN_NAME,
    tags    = ['ppo', '15k', 'produccion'],
    config  = trainer_config,
)

# ============ ENTRENAR ============
cstr    = CSTRSimulator(dt=1.0, control_limits=((300, 420), (99.5, 104)))
trainer = PPOTrainer(trainer_config)
trainer.env.proceso.connect_external_process(cstr)
trainer.train()

# ============ MÉTRICAS FINALES ============
wandb.log({
    'final_eval_reward'         : trainer.best_reward,
    'total_episodes'            : len(trainer.episode_rewards),
    'final_reward_mean10'       : np.mean(trainer.episode_rewards[-10:]),
    'final_energy_mean10'       : np.mean(trainer.episode_energies[-10:]),
    'final_overshoot_mean10'    : np.mean(trainer.episode_max_overshoots[-10:]),
    'final_actor_loss_mean10'   : np.mean(trainer.actor_losses[-10:])  if trainer.actor_losses  else 0,
    'final_critic_loss_mean10'  : np.mean(trainer.critic_losses[-10:]) if trainer.critic_losses else 0,
    'final_clip_frac_mean10'    : np.mean(trainer.clip_fractions[-10:]) if hasattr(trainer, 'clip_fractions') and trainer.clip_fractions else 0,
}, step=len(trainer.episode_rewards))

wandb.finish()
print(f'Run completado: {RUN_NAME}')

## 7. Prueba de rendimiento

In [ ]:
DEVICE             = 'cuda' if torch.cuda.is_available() else 'cpu'
MANIPULABLE_RANGES = [(300, 420), (99.5, 104)]
PPO_CHECKPOINT = 'checkpoints/ppo_ctrl_15k_prod/agent_ctrl_best.pt'  # ← ajustar path

eval_config = {
    'env_config': {
        'architecture'           : 'simple',
        'env_type'               : 'simulation',
        'action_type'            : 'continuous',
        'n_manipulable_vars'     : 2,
        'manipulable_ranges'     : [(300, 420), (99.5, 104)],
        'manipulable_setpoints'  : None,
        'dt_usuario'             : 1.0,
        'max_steps'              : 100,
        'max_time_detector'      : 60,
        'reward_dead_band'       : 0.02,
        'delta_percent_ctrl'     : 0.2,
        'reward_weights'         : {'error': 1.0, 'tiempo': 0.001, 'overshoot': 0.3, 'energy': 0.001},
        'pid_limits'             : [(0.01, 50.0), (0.001, 1.0), (0.0, 1.0)],
        'agent_controller_config': {'agent_type': 'continuous'},
        'env_type_config'        : {'dt': 1.0, 'control_limits': ((300, 420), (99.5, 104))},
        'stability_config'       : {
            'error_increase_tolerance': 2.0,
            'max_sign_changes_ratio'  : 0.3,
            'max_abrupt_change_ratio' : 0.05,
            'abrupt_change_threshold' : 0.2,
        },
    },
    'agent_ctrl_config': {
        'algorithm'    : 'ppo',
        'state_dim'    : 10,
        'action_dim'   : 6,
        'n_vars'       : 2,
        'action_type'  : 'continuous',
        'hidden_dims'  : (64, 32),
        'lr_actor'     : 3e-04,
        'lr_critic'    : 1e-03,
        'gamma'        : 0.99,
        'clip_epsilon' : 0.2,
        'ppo_epochs'     : 8,
        'gae_lambda'   : 0.95,
        'entropy_coef' : 0.01,
        'value_coef'   : 0.5,
        'mini_batch_size'   : 64,
        'rollout_steps': 512,
        'device'       : DEVICE,
        'seed'         : 42,
    },
    'n_episodes'    : 1,
    'use_wandb'     : False,
    'checkpoint_dir': 'checkpoints/eval_tmp',
}
cstr = CSTRSimulator(dt=1.0, control_limits=(MANIPULABLE_RANGES[0], MANIPULABLE_RANGES[1]))
trainer = PPOTrainer(eval_config)
trainer.env.proceso.connect_external_process(cstr)
trainer.agent_ctrl.load(PPO_CHECKPOINT)
print('Agente PPO cargado correctamente')

## 8. Pruebas con otros SP

In [ ]:
# ============ GRILLA DE SETPOINTS ============
T_setpoints = [320, 340, 360, 380, 400]   # K
V_setpoints = [100.0, 101.0, 102.0]       # m³

# ============ FUNCIÓN DE EVALUACIÓN ============
def evaluar_sp(trainer, T_sp, V_sp, max_steps=100):
    state = trainer.env.reset()[0]
    trainer.env.manipulable_setpoints = [T_sp, V_sp]
    trainer.env._update_errors()
    state = trainer.env._get_observation()

    print(f'SP después del reset: {trainer.env.manipulable_setpoints}')

    T_hist, V_hist = [], []
    done = False
    step = 0

    while not done and step < max_steps:
        action = trainer.agent_ctrl.select_action(state, training=False)
        next_state, reward, terminated, truncated, info = trainer.env.step(action)
        done = terminated or truncated

        T_hist.append(trainer.env.manipulable_pvs[0])
        V_hist.append(trainer.env.manipulable_pvs[1])

        state = next_state
        step += 1

    return np.array(T_hist), np.array(V_hist)

In [ ]:
# ============ EVALUAR Y GRAFICAR — VARIANDO T ============
fig, axes = plt.subplots(
    len(T_setpoints), 2,
    figsize=(14, 4 * len(T_setpoints))
)

for i, T_sp in enumerate(T_setpoints):
    V_sp = 101.0

    T_hist, V_hist = evaluar_sp(trainer, T_sp, V_sp)
    steps = np.arange(1, len(T_hist) + 1)

    ax_T = axes[i, 0]
    ax_T.plot(steps, T_hist, color='steelblue', linewidth=2, label='T (PV)')
    ax_T.axhline(T_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_sp}K')
    ax_T.set_title(f'Temperatura — SP={T_sp}K, V_sp={V_sp}m³')
    ax_T.set_xlabel('Step')
    ax_T.set_ylabel('T (K)')
    ax_T.legend()
    ax_T.grid(True, alpha=0.3)

    ax_V = axes[i, 1]
    ax_V.plot(steps, V_hist, color='orange', linewidth=2, label='V (PV)')
    ax_V.axhline(V_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={V_sp}m³')
    ax_V.set_title(f'Volumen — SP={T_sp}K, V_sp={V_sp}m³')
    ax_V.set_xlabel('Step')
    ax_V.set_ylabel('V (m³)')
    ax_V.legend()
    ax_V.grid(True, alpha=0.3)

    T_final = T_hist[-1]
    V_final = V_hist[-1]
    print(f'T_sp={T_sp}K → T_final={T_final:.2f}K (error={abs(T_final-T_sp):.2f}K) | '
          f'V_final={V_final:.3f}m³ (error={abs(V_final-V_sp):.4f}m³)')

plt.suptitle('Evaluación PPO CTRL — Distintos Setpoints de T', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eval_ppo_ctrl_T_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============ EVALUAR Y GRAFICAR — VARIANDO V ============
fig, axes = plt.subplots(
    len(V_setpoints), 2,
    figsize=(14, 4 * len(V_setpoints))
)

T_sp_fijo = 340.0

for i, V_sp in enumerate(V_setpoints):
    T_hist, V_hist = evaluar_sp(trainer, T_sp_fijo, V_sp)
    steps = np.arange(1, len(T_hist) + 1)

    ax_T = axes[i, 0]
    ax_T.plot(steps, T_hist, color='steelblue', linewidth=2)
    ax_T.axhline(T_sp_fijo, color='red', linestyle='--', linewidth=1.5, label=f'SP={T_sp_fijo}K')
    ax_T.set_title(f'Temperatura — T_sp={T_sp_fijo}K, V_sp={V_sp}m³')
    ax_T.set_xlabel('Step')
    ax_T.set_ylabel('T (K)')
    ax_T.legend()
    ax_T.grid(True, alpha=0.3)

    ax_V = axes[i, 1]
    ax_V.plot(steps, V_hist, color='orange', linewidth=2)
    ax_V.axhline(V_sp, color='red', linestyle='--', linewidth=1.5, label=f'SP={V_sp}m³')
    ax_V.set_title(f'Volumen — T_sp={T_sp_fijo}K, V_sp={V_sp}m³')
    ax_V.set_xlabel('Step')
    ax_V.set_ylabel('V (m³)')
    ax_V.legend()
    ax_V.grid(True, alpha=0.3)

    T_final = T_hist[-1]
    V_final = V_hist[-1]
    print(f'V_sp={V_sp}m³ → V_final={V_final:.3f}m³ (error={abs(V_final-V_sp):.4f}m³) | '
          f'T_final={T_final:.2f}K (error={abs(T_final-T_sp_fijo):.2f}K)')

plt.suptitle('Evaluación PPO CTRL — Distintos Setpoints de V', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eval_ppo_ctrl_V_setpoints.png', dpi=150, bbox_inches='tight')
plt.show()